# 🏈 NFL PlayVision
### Real tracking · rolling motion forecasts · interactive receiver–defender replay

Buka `PlayVision_Offline.html` langsung untuk pengalaman offline tanpa instalasi. Notebook ini menampilkan hasil terverifikasi dan dapat melatih ulang model.

**Lingkup:** prediksi posisi 0,5/1 detik, bukan hasil pertandingan atau catch probability. Data model bawaan berasal dari 12 pertandingan 2021. Tiga replay lengkap 2017 merupakan ilustrasi dari era lain. Data 2026 Kaggle belum dilatih di sini.

In [ ]:
from pathlib import Path
import sys,os,zipfile
IN_COLAB='google.colab' in sys.modules or Path('/content').exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH=Path('/content/drive/MyDrive/NFL_PlayVision_v2.zip')
    if not ZIP_PATH.exists():raise FileNotFoundError('Upload NFL_PlayVision_v2.zip ke My Drive, atau ubah ZIP_PATH.')
    destination=Path('/content/playvision_workspace');destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        for name in z.namelist():
            if not (destination/name).resolve().is_relative_to(destination.resolve()):raise ValueError('Unsafe ZIP path')
        z.extractall(destination)
    PROJECT=destination/'nfl_playvision'
else:
    PROJECT=Path.cwd()
    if not (PROJECT/'src/playvision.py').exists():PROJECT=PROJECT/'nfl_playvision'
PROJECT=PROJECT.resolve()
assert (PROJECT/'src/playvision.py').exists()
os.chdir(PROJECT)
print('Project:',PROJECT)

In [ ]:
import subprocess
if IN_COLAB:subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
# Jika diminta restart runtime, restart lalu jalankan kembali sel setup.
TRAIN_NEW=False
RUN_TESTS=False
BUDGET_PER_AXIS=60
MAX_TRAIN_SAMPLES=100000
import json,datetime,shutil
import pandas as pd
import plotly.express as px
from IPython.display import display,HTML,Markdown
sys.path.insert(0,str(PROJECT/'src'))
from playvision import download,prepare,train,evaluate,predict
RUN=PROJECT/'models/hybrid_v2' 

## 1 · Replay offline
Semua posisi berasal dari tracking nyata. Future truth hanya overlay evaluasi, tidak menjadi input model. Panel nearest DB adalah hubungan geometris, bukan assignment coverage.

Aplikasi berikut adalah demo bawaan; sumber adegan selalu ditampilkan. Jika tampilan iframe dibatasi oleh browser Colab, unduh dan buka HTML langsung.

In [ ]:
import html
page=(PROJECT/'PlayVision_Offline.html').read_text(encoding='utf-8')
frame='<iframe title="NFL PlayVision" style="width:100%;height:1100px;border:0" sandbox="allow-scripts" srcdoc="'+html.escape(page,quote=True)+'"></iframe>'
display(HTML(frame))

## 2 · Data dan split
Delapan game latihan minggu 1–2, dua validasi minggu 3, dua tes minggu 4. Tidak ada game yang muncul di lebih dari satu split. Frame berdekatan tetap berkorelasi, sehingga jumlah frame bukan jumlah pertandingan independen.

Regional data berakhir 0,3–0,5 detik setelah release; angka regional terutama menggambarkan fase sebelum release dalam passing play. Lihat phase metrics dan evaluasi 2017 terpisah.

In [ ]:
plan=json.loads((PROJECT/'data/split_plan.json').read_text())
display(pd.DataFrame([{'gameId':g,'split':s} for g,s in plan.items()]))
if RUN_TESTS:
    subprocess.run([sys.executable,'-m','pytest','tests','-q'],check=True)

## 3 · Pelatihan ulang (opsional)
Fitur menggunakan riwayat dan tetangga pada frame saat ini saja. AutoML belajar koreksi dx/dy terhadap baseline kecepatan tetap. Validasi memilih baseline/campuran untuk setiap horizon. Test dibuka setelah model dibekukan.

Mengulang pelatihan pada split tes yang sama tidak menghasilkan tes independen baru. CPU cukup. Budget bukan batas waktu total keras.

In [ ]:
if TRAIN_NEW:
    download();prepare()
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    RUN=PROJECT/'runs'/f'colab_{stamp}'
    train(BUDGET_PER_AXIS,MAX_TRAIN_SAMPLES,run_dir=RUN)
    evaluate(run_dir=RUN)
    if IN_COLAB:
        saved=Path('/content/drive/MyDrive/NFL_PlayVision_runs')/RUN.name
        shutil.copytree(RUN,saved,dirs_exist_ok=True)
        shutil.copytree(PROJECT/'data',Path('/content/drive/MyDrive/NFL_PlayVision_runs/data_snapshot'),dirs_exist_ok=True)
        print('Saved:',saved)
print('Using metrics:',RUN)

## 4 · Ukuran hasil yang bermakna
Error diukur dalam yards. Persentase dalam toleransi 1 yard bukan akurasi kemenangan, catch, atau produksi mingguan. Bandingkan pula baseline percepatan tetap yang lebih kuat.

In [ ]:
metrics=json.loads((RUN/'metrics.json').read_text());rows=[]
for h,record in metrics['horizons'].items():
    for name in ['constant_velocity','constant_acceleration','selected_model']:
        m=record[name]
        rows.append({'horizon_seconds':h,'model':name,'n':m['n'],'mean_error_yards':m['mean_distance_yards'],
           'within_half_yard_pct':m['within_0_5_yards_pct'],'within_one_yard_pct':m['within_1_yard_pct']})
summary=pd.DataFrame(rows);display(summary.round(3))
px.bar(summary,x='horizon_seconds',y='mean_error_yards',color='model',barmode='group',template='plotly_white',title='Held-out game error: lower is better').show()
print('Test games:',metrics['test_games'],'plays:',metrics['test_plays'])

## 5 · Inferensi tanpa koordinat masa depan
Contoh berisi fitur yang sudah dihitung dari frame saat ini dan sejarahnya. `build_inputs` dapat membuat fitur dari prefix observasi saja tanpa membutuhkan future labels.

In [ ]:
import joblib
bundle=joblib.load(RUN/'model.joblib')
example=pd.read_csv(PROJECT/'examples/forecast_inputs.csv')
dx,dy=predict(bundle,example)
example['predicted_delta_x']=dx;example['predicted_delta_y']=dy
display(example[['gameId','playId','actor_id','frameId','horizon','predicted_delta_x','predicted_delta_y']].head())

## 6 · Evaluasi eksternal dan batasan
Angka berikut adalah hasil model bawaan pada satu game 2017, termasuk fase setelah release. Bukan angka run baru jika Anda retrain. Perbedaan era dan satu game membatasi klaim generalisasi.

Untuk data Kaggle 2026, baca KAGGLE_2026_GUIDE.md. Importer membuang future landing coordinates dan role target. Adapter schema diuji dengan fixture, belum dengan dataset kompetisi asli.

In [ ]:
external=json.loads((PROJECT/'reports/hybrid_v2_external2017.json').read_text())
display(pd.DataFrame([{'horizon':h,'v1_post_release_error':m['post_release']['v1']['mean_distance_yards'],'v2_post_release_error':m['post_release']['v2']['mean_distance_yards']} for h,m in external['horizons'].items()]))
display(Markdown((PROJECT/'reports/RESULTS.md').read_text(encoding='utf-8')))
if IN_COLAB:
    save_root=Path('/content/drive/MyDrive/NFL_PlayVision_runs');save_root.mkdir(exist_ok=True)
    summary.to_csv(save_root/'metrics_summary.csv',index=False)
    shutil.copy2(PROJECT/'PlayVision_Offline.html',save_root/'PlayVision_Offline.html')
    print('Saved:',save_root)